# Sentiment Analysis of Airline Tweets
Imports and setup.

In [5]:
import os
import re
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

for p in ['stopwords','wordnet','vader_lexicon']:
    nltk.download(p, quiet=True)

os.makedirs('outputs', exist_ok=True)
sns.set_style('whitegrid')

Load the tweets and take a first look.

In [6]:
df = pd.read_csv('data/Tweets.csv')
print(df.shape)
df.head()

(14640, 15)


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [7]:
print(df['airline_sentiment'].value_counts())
print(df['airline'].value_counts())


print(df.isna().sum())

airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64
airline
United            3822
US Airways        2913
American          2759
Southwest         2420
Delta             2222
Virgin America     504
Name: count, dtype: int64
tweet_id                            0
airline_sentiment                   0
airline_sentiment_confidence        0
negativereason                   5462
negativereason_confidence        4118
airline                             0
airline_sentiment_gold          14600
name                                0
negativereason_gold             14608
retweet_count                       0
text                                0
tweet_coord                     13621
tweet_created                       0
tweet_location                   4733
user_timezone                    4820
dtype: int64


Clean the table. Keep useful columns, remove duplicates and fix the dates.

In [8]:
# keep only the columns we need
df = df[['tweet_id','airline','airline_sentiment','negativereason','text','tweet_created','user_timezone','retweet_count']].copy()

df = df.drop_duplicates(subset='tweet_id')
df=df.dropna(subset=['text','airline_sentiment'])
df = df.drop_duplicates(subset='text')     # same text posted twice

df['tweet_created'] = pd.to_datetime(df['tweet_created'], utc=True)
df['date']=df['tweet_created'].dt.date
df['user_timezone'] = df['user_timezone'].fillna('Unknown')

print(df.shape)
print(df['date'].min(), df['date'].max())

(14427, 9)
2015-02-17 2015-02-24
